# Solution


In [1]:
from typing import override
from pprint import pprint

from systrade import (
    BacktestBroker,
    BarData,
    Engine,
    ExecutionReport,
    FileFeed,
    Strategy,
)

## Strategy implementation


In [2]:
class MomentumStrategy(Strategy):
    """A long only momentum strategy which will buy all you can afford if:

     1. Current closing price > yesterday's closing price > 2 days ago's closing price
     2. Is not currently invested in the security

    and  will close the position if:

     1. Current closing price < yesterday's closing price < 2 days ago's closing price
     2. Is currently invested in the security

    """

    def __init__(self, symbol: str) -> None:
        super().__init__()
        self.symbol = symbol
        # To keep track of historical prices
        self.history = []

    @override
    def on_start(self) -> None:
        self.subscribe(self.symbol)

    @override
    def on_data(self, data: BarData) -> None:
        # Save the latest prices
        price = data[self.symbol].close
        # Compute signal
        if len(self.history) >= 2:
            buy_signal = price > self.history[-1] > self.history[-2]
            sell_signal = price < self.history[-1] < self.history[-2]
            # Act on signal
            if buy_signal and not self.portfolio.is_invested_in(self.symbol):
                # Try to buy all we can afford
                qty = self.portfolio.cash() / price
                self.post_market_order(self.symbol, qty)
            elif sell_signal and self.portfolio.is_invested_in(self.symbol):
                # Close out current position
                pos = self.portfolio.position(self.symbol)
                self.post_market_order(self.symbol, -pos.qty)

        # Save for next round
        self.history.append(price)

    @override
    def on_execution(self, report: ExecutionReport) -> None:
        # When we get a fill this event will be triggered
        print("Notified of execution:")
        pprint(report, indent=2)

## Run backtest


I'll use entire history here.


In [3]:
cash = 1000
feed = FileFeed("history.csv")
broker = BacktestBroker()
strategy = MomentumStrategy("NVDA")
engine = Engine(feed, broker, strategy, cash)
engine.run()

Notified of execution:
ExecutionReport(order=Order(id='1',
                            symbol='NVDA',
                            quantity=np.float64(6084.510393074551),
                            type=<OrderType.MARKET: 1>,
                            submit_time=datetime.datetime(2005, 1, 14, 0, 0, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>),
                            price=None),
                last_price=np.float64(0.1625179515781858),
                last_quantity=np.float64(6084.510393074551),
                cum_quantity=np.float64(6084.510393074551),
                rem_quantity=0.0,
                fill_timestamp=datetime.datetime(2005, 1, 18, 0, 0, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>))
Notified of execution:
ExecutionReport(order=Order(id='2',
                            symbol='NVDA',
                            quantity=np.float64(-6084.510393074551),
                            type=<OrderType.MARKET: 1>,
                   

## Results


Let's see how this strategy compares to buying and holding the S&P vs. buying
and holding on the same underlying.


In [4]:
activity = engine.portfolio.activity()

In [5]:
total_return_nvda_momentum = activity.total_return()
total_return_nvda_momentum

np.float64(14.292038439601408)

Our strategy achieved 1,430% returns over the entire trading period.


Let's grab the entire history for NVDA and the S&P to look more into it.


In [6]:
close_nvda = feed.df.loc["NVDA"]["Close"]
close_gspc = feed.df.loc["^GSPC"]["Close"]

How long was our history?


In [7]:
total_trading_days = len(close_nvda)
print(total_trading_days / 252)

20.626984126984127


Over 20 years ~1,430% is pretty solid. Annually this is


In [8]:
ann_return_nvda_momentum = (1 + total_return_nvda_momentum) ** (
    252 / total_trading_days
) - 1
ann_return_nvda_momentum

np.float64(0.14136118839925982)

or ~14% annually. How does that compare to buy and hold with market returns?


In [9]:
ann_return_gspc = (close_gspc.iloc[-1] / close_gspc.iloc[0]) ** (
    252 / total_trading_days
) - 1
ann_return_gspc

np.float64(0.08494040883382747)

We've actually achieved significant excess return with this strategy over the
annualized market return of ~8.5%.


What if we just bought and hold NVDA though?


In [10]:
close_nvda.iloc[-1] / close_nvda.iloc[0]

np.float64(966.7097557082327)

Wow, ~96,600% returns! Annually this is


In [11]:
ann_return_nvda = (close_nvda.iloc[-1] / close_nvda.iloc[0]) ** (
    252 / total_trading_days
) - 1
ann_return_nvda

np.float64(0.3954931413615359)

almost 40%.


Although we outperformed the market, we'd have been much better off just buying
and holding NVDA. A more comprehensive analysis would consider risk and
efficiency metrics such as drawdown and Sharpe ratio.
